# Physical AI 最終課題 — 障害物の追加・再学習の比較

このNotebookは**Google ColabのCPU**で動作する、Q-learningによる2D格子ロボットのシミュレーションです。Docker・ROS・Gazebo・GPUは不要です。

## 実験の順番（自動実行）
1. 元の障害物12個の環境でAIを学習
2. `OBSTACLES.add((3, 6))` で障害物を追加し、**元のAI**と**再学習AI**を比較
3. 元の経路にある `(6, 2)` に障害物を追加する実験も行い、再学習効果を比較
4. 112秒の動画、結果CSV、PDFを生成

**実行方法：上部メニュー「ランタイム」→「すべてのセルを実行」。**

注意：(3,6)は元のAIの経路外なので結果は大きく変わらないことがあります。物理エンジンや実機のロボットではなく、離散2D格子の簡易シミュレーションです。

In [ ]:
# 初回だけパッケージを準備します。ColabではGPUは不要です。
%pip -q install numpy pillow reportlab
import shutil
print('FFmpeg:', shutil.which('ffmpeg'))
if shutil.which('ffmpeg') is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg

## 1. 学習→障害物追加→比較実験→動画生成
ここで保存する元のAIは上書きせずに保持します。変更した環境で元のAIを評価してから、別のAIを再学習します。実験結果は `results/` に保存されます。

In [ ]:
"""Physical AI: reproducible grid robot Q-learning comparison.

Run: python run_experiment.py
Colab: no GPU, ROS, Docker required. Simulated discrete grid, not a physics engine.
All scenarios are trained from scratch unless explicitly called 'original policy'.
"""
from __future__ import annotations

import csv
import json
import shutil
import subprocess
import time
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFont

SIZE = 8
START = (0, 0)
GOAL = (7, 7)
SEED = 2026
TRAIN_EPISODES = 6000
MAX_TRAIN_STEPS = 80
MAX_TEST_STEPS = 40
ACTIONS = ((-1, 0), (0, 1), (1, 0), (0, -1))
BASE_OBSTACLES = frozenset({
    (1,1), (1,2), (1,3), (2,3), (3,3), (4,3),
    (5,1), (5,2), (5,3), (4,5), (5,5), (6,5)
})
# Experiments requested by the user + an additional on-route obstruction.
# This is equivalent to adding OBSTACLES.add((3, 6)) *after*
# the original model has been trained; use a separate copy to preserve the baseline.
OBSTACLES = set(BASE_OBSTACLES)
OBSTACLES.add((3, 6))
OBSTACLES_OFF_ROUTE = frozenset(OBSTACLES)

# Extra test: add an obstacle on the original learned route.
OBSTACLES = set(BASE_OBSTACLES)
OBSTACLES.add((6, 2))
OBSTACLES_ON_ROUTE = frozenset(OBSTACLES)
OUTPUT = Path('results')


def step(pos, action, obstacles):
    dr, dc = ACTIONS[action]
    nxt = (pos[0]+dr, pos[1]+dc)
    blocked = (not 0 <= nxt[0] < SIZE or not 0 <= nxt[1] < SIZE or nxt in obstacles)
    if blocked:
        return pos, -1.0, False, True
    if nxt == GOAL:
        return nxt, 10.0, True, False
    return nxt, -0.08, False, False


def train(obstacles, seed=SEED, episodes=TRAIN_EPISODES):
    rng = np.random.default_rng(seed)
    q = np.zeros((SIZE, SIZE, len(ACTIONS)), dtype=np.float64)
    train_success = []
    t0 = time.perf_counter()
    for episode in range(episodes):
        pos, reached = START, False
        epsilon = max(0.04, 0.95 * 0.9985 ** episode)
        for _ in range(MAX_TRAIN_STEPS):
            a = int(rng.integers(4)) if rng.random() < epsilon else int(np.argmax(q[pos]))
            nxt, reward, done, _ = step(pos, a, obstacles)
            target = reward if done else reward + 0.95 * np.max(q[nxt])
            q[pos][a] += 0.22 * (target - q[pos][a])
            pos = nxt
            if done:
                reached = True
                break
        train_success.append(reached)
    return q, train_success, time.perf_counter() - t0


def run_episode(q, obstacles, *, seed=0, max_steps=MAX_TEST_STEPS):
    rng = np.random.default_rng(seed)
    pos = START
    states, actions, blocked_count = [pos], [], 0
    for _ in range(max_steps):
        a = int(np.argmax(q[pos])) if q is not None else int(rng.integers(4))
        pos, _, done, blocked = step(pos, a, obstacles)
        states.append(pos)
        actions.append(a)
        blocked_count += int(blocked)
        if done:
            break
    return dict(states=states, actions=actions, success=(pos==GOAL),
                steps=len(actions), blocked=blocked_count)


def evaluate(q, obstacles, trials=100):
    runs = [run_episode(q, obstacles, seed=1000+i) for i in range(trials)]
    return {
        'success_rate_percent': 100 * sum(x['success'] for x in runs) / trials,
        'mean_steps': sum(x['steps'] for x in runs) / trials,
        'mean_blocked_moves': sum(x['blocked'] for x in runs) / trials,
    }


def font(size, bold=False):
    name = 'DejaVuSans-Bold.ttf' if bold else 'DejaVuSans.ttf'
    places = [f'/usr/share/fonts/truetype/dejavu/{name}',
              '/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf']
    for path in places:
        if Path(path).exists():
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

FB = font(31, True)
FM = font(21, True)
FS = font(18)
FT = font(15)


def draw_map(d, obstacles, result, count, *, origin=(38, 111), cell=49):
    ox, oy = origin
    def xy(p):
        return (int(ox+p[1]*cell+cell/2-2), int(oy+p[0]*cell+cell/2-2))
    for r in range(SIZE):
        for c in range(SIZE):
            x, y = ox+c*cell, oy+r*cell
            fill = (148, 80, 72) if (r,c) in obstacles else (29, 46, 65)
            d.rounded_rectangle((x,y,x+cell-4,y+cell-4), radius=5,
                                fill=fill, outline=(76, 96, 116))
    gx, gy = xy(GOAL)
    d.ellipse((gx-16,gy-16,gx+16,gy+16), fill=(40,173,128),outline='white',width=2)
    d.text((gx-8,gy-12),'G',fill='white',font=FM)
    n = min(len(result['actions']),max(0,count))
    states = result['states'][:n+1]
    if len(states)>1:
        d.line([xy(p) for p in states],fill=(85,222,173),width=5)
    x, y=xy(result['states'][n])
    d.ellipse((x-18,y-18,x+18,y+18), fill=(54,156,236),outline='white',width=3)
    d.ellipse((x-4,y-4,x+4,y+4), fill='white')
    return n


def draw_frame(t, scenario_data):
    # 112 seconds, 6fps. More than one minute and below three minutes.
    stages = [
        (0,10,'Physical AI: learning to navigate','Overview',0),
        (10,30,'A: original map / trained AI','original',1),
        (30,50,'B: add (3,6) / SAME AI','off_old',2),
        (50,73,'C: add (6,2) / SAME AI','on_old',3),
        (73,96,'D: add (6,2) / RETRAINED AI','on_new',4),
        (96,112,'Results and limitations','summary',5),
    ]
    start, end, title, key, stage_n = next(x for x in stages if x[0] <= t < x[1])
    w, h = 960, 540
    img=Image.new('RGB',(w,h),(13,21,35))
    d=ImageDraw.Draw(img)
    d.text((33,19),'Physical AI | Q-learning navigation',fill=(244,249,255),font=FB)
    d.text((33,61),'2D grid simulation / CPU / 6,000 training episodes',fill=(175,195,216),font=FS)
    if key=='summary':
        s=scenario_data
        rows=[
            ('Original map + learned AI',s['original']['eval']['success_rate_percent']),
            ('Add (3,6) + old AI',s['off_old']['eval']['success_rate_percent']),
            ('Add (3,6) + retrained AI',s['off_new']['eval']['success_rate_percent']),
            ('Add (6,2) + old AI',s['on_old']['eval']['success_rate_percent']),
            ('Add (6,2) + retrained AI',s['on_new']['eval']['success_rate_percent']),
        ]
        d.text((40,110),'Goal success in fixed-start repeated tests',fill='white',font=FM)
        for i,(label,rate) in enumerate(rows):
            y=160+i*56
            d.text((48,y),label,fill=(230,238,246),font=FS)
            d.rectangle((435,y+2,435+round(rate)*3.7,y+24),fill=(46,178,138) if rate else (215,114,93))
            d.text((825,y),f'{rate:.0f}%',fill='white',font=FS)
        d.text((42,471),'Caution: repeated deterministic runs, not unseen-map generalization.',fill=(235,192,110),font=FT)
    else:
        scene_key='original' if key=='Overview' else key
        sc=scenario_data[scene_key]
        obstacles=sc['obstacles']
        run=sc['run']
        n = 0 if stage_n==0 else int((t-start)*1.05)
        n=draw_map(d,obstacles,run,n)
        d.text((460,118),title,fill=(237,245,253),font=FM)
        d.text((460,168),f'Obstacles: {len(obstacles)}',fill=(189,205,226),font=FS)
        d.text((460,210),f'Actions shown: {n} / {len(run["actions"])}',fill=(189,205,226),font=FS)
        d.text((460,250),f'Goal reached: {"YES" if n==len(run["actions"]) and run["success"] else "NO / in progress"}',fill=(198,226,232),font=FS)
        d.text((460,295),f'Blocked moves (final): {run["blocked"]}',fill=(189,205,226),font=FS)
        if key == 'Overview':
            note=['Study question: does a learned policy', 'adapt when the obstacle map changes?', 'Compare without and with retraining.']
        elif key == 'off_old':
            note=['Added (3,6) is NOT on the initial path.', 'An unchanged outcome is still a result.']
        elif key == 'on_old':
            note=['Added (6,2) blocks the initial path.', 'Old policy can keep trying a blocked cell.']
        elif key == 'on_new':
            note=['Retrain in the new environment.', 'New policy may discover an alternate route.']
        else:
            note=['Original policy on the original map.', 'Shortest path from (0,0) to (7,7):', 'at least 14 moves without diagonal actions.']
        for j,line in enumerate(note):
            d.text((460,354+j*29),line,fill=(122,219,181) if j==0 else (197,208,224),font=FT)
    d.line((33,508,927,508),fill=(62,85,106),width=2)
    d.text((35,518),'Discrete grid robot, NOT real hardware or a physics-engine simulation',fill=(156,170,189),font=FT)
    return np.asarray(img, dtype=np.uint8)


def make_video(scenarios, dest=OUTPUT/'comparison_demo.mp4', fps=6, secs=112):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg missing. In Colab: !apt-get -qq install ffmpeg')
    cmd=['ffmpeg','-y','-hide_banner','-loglevel','error',
         '-f','rawvideo','-vcodec','rawvideo','-s','960x540','-pix_fmt','rgb24',
         '-r',str(fps),'-i','-','-an','-vcodec','libx264','-preset','ultrafast',
         '-crf','25','-pix_fmt','yuv420p','-movflags','+faststart',str(dest)]
    p=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    try:
        for j in range(secs*fps):
            p.stdin.write(draw_frame(j/fps,scenarios).tobytes())
        p.stdin.close()
        err=p.stderr.read().decode(errors='replace')
        status=p.wait()
        if status != 0:
            raise RuntimeError(f'ffmpeg exited {status}: {err}')
    except Exception:
        if p.poll() is None:p.kill()
        raise
    return dest


def make_route_image(scenarios, dest=OUTPUT/'route_comparison.png'):
    img=Image.new('RGB',(1200,525),(13,21,35))
    d=ImageDraw.Draw(img)
    keys=['original','off_old','on_old']
    titles=['A ORIGINAL 12','B ADD (3,6) 13','C ADD (6,2) 13']
    for j,(key,title) in enumerate(zip(keys,titles)):
        sc=scenarios[key]
        x0=j*400
        d.text((x0+22,24),title,fill=(234,243,253),font=FM)
        draw_map(d,sc['obstacles'],sc['run'],len(sc['run']['actions']),origin=(x0+13,77),cell=44)
        d.text((x0+23,463),'SUCCESS' if sc['run']['success'] else 'FAILED',fill=(67,220,152) if sc['run']['success'] else (240,130,111),font=FS)
    img.save(dest)
    return dest


def main():
    OUTPUT.mkdir(parents=True, exist_ok=True)
    base_q, base_history, t_base = train(BASE_OBSTACLES)
    off_q, off_history, t_off = train(OBSTACLES_OFF_ROUTE)
    on_q, on_history, t_on = train(OBSTACLES_ON_ROUTE)
    np.save(OUTPUT/'q_original.npy',base_q)
    np.save(OUTPUT/'q_with_3_6.npy',off_q)
    np.save(OUTPUT/'q_with_6_2.npy',on_q)

    specs=[
        ('random','Original environment / random','original',BASE_OBSTACLES,None),
        ('original','Original environment / trained AI','original',BASE_OBSTACLES,base_q),
        ('off_old','Add (3,6) / original AI','off',OBSTACLES_OFF_ROUTE,base_q),
        ('off_new','Add (3,6) / retrained AI','off',OBSTACLES_OFF_ROUTE,off_q),
        ('on_old','Add (6,2) / original AI','on',OBSTACLES_ON_ROUTE,base_q),
        ('on_new','Add (6,2) / retrained AI','on',OBSTACLES_ON_ROUTE,on_q),
    ]
    scenarios={}
    rows=[]
    for key,label,kind,obstacles,policy in specs:
        run=run_episode(policy,obstacles,seed=1234 if policy is None else 0)
        ev=evaluate(policy,obstacles)
        rows.append(dict(condition=label, obstacles=len(obstacles),
                         success_rate_percent=ev['success_rate_percent'],
                         mean_steps=round(ev['mean_steps'],1),
                         mean_blocked_moves=round(ev['mean_blocked_moves'],1),
                         one_run_success=int(run['success']),
                         one_run_steps=run['steps'],one_run_blocked=run['blocked']))
        scenarios[key]=dict(label=label,obstacles=obstacles,run=run,eval=ev)
    with (OUTPUT/'comparison_results.csv').open('w',newline='',encoding='utf-8-sig') as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0]))
        wr.writeheader();wr.writerows(rows)
    path_orig=scenarios['original']['run']['states']
    report={
        'seed':SEED, 'train_episodes':TRAIN_EPISODES,
        'train_time_seconds':{'original':t_base,'add_3_6':t_off,'add_6_2':t_on},
        'training_success_last_100_pct':{
            'original':sum(base_history[-100:]),
            'add_3_6':sum(off_history[-100:]),
            'add_6_2':sum(on_history[-100:])},
        'original_route':[list(x) for x in path_orig],
        'does_original_route_use_3_6':(3,6) in path_orig,
        'does_original_route_use_6_2':(6,2) in path_orig,
        'fixed_start_goal':True,
        'deterministic_ai_repeated_100_times':True,
        'scenario_results':rows,
        'limitations':'Discrete 2D grid only. 100 repeated deterministic policy evaluations with same start and goal are NOT 100 independent trials; no physics engine or real robot.'
    }
    (OUTPUT/'summary.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
    txt=['Physical AI - Q-learning obstacles comparison',
         f'Seed={SEED}; training episodes={TRAIN_EPISODES}',
         'Obstacle (3,6) on old path? '+str((3,6) in path_orig),
         'Obstacle (6,2) on old path? '+str((6,2) in path_orig),'']
    for r in rows:
        txt.append(f"{r['condition']}: success={r['success_rate_percent']:.1f}%, steps={r['one_run_steps']}, blocked={r['one_run_blocked']}")
    txt+=['','Note: 100 repeats of the deterministic greedy policy use the SAME fixed start/goal.','This is a simplified grid simulation, NOT a physical-engine or hardware experiment.']
    (OUTPUT/'results.txt').write_text('\n'.join(txt)+'\n',encoding='utf-8')
    make_route_image(scenarios)
    print('\n'.join(txt))
    print('Creating 112-second comparison video...')
    p=make_video(scenarios)
    print('Created:',p,'bytes:',p.stat().st_size)
    return report

if __name__ == '__main__':
    main()


## 2. PDFレポートの自動生成
次のセルの上部にある `OMNI_ACCOUNT`、`GITHUB_URL`、`YOUTUBE_URL` を、ご自身の情報に編集してから実行してください。未入力でも仮のPDFが作られます。

In [ ]:
"""Create Japanese submission PDF and editable Markdown from measured results.
Run after run_experiment.py: python make_report.py
"""
import json
from pathlib import Path
from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER
from reportlab.lib.units import mm
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak, KeepTogether

OUTPUT = Path('results')
# EDIT THESE THREE VALUES BEFORE SUBMITTING (no credentials are needed).
OMNI_ACCOUNT = '提出前にアカウント名へ変更'
GITHUB_URL = '［提出前にURLを記入］'
YOUTUBE_URL = '［新しい比較動画をアップロード後にURLを記入］'
OUTPDF = Path(f'説明資料_PAI最終課題_{OMNI_ACCOUNT}.pdf')
OUTMD = Path('レポート文章_編集用.md')

pdfmetrics.registerFont(UnicodeCIDFont('HeiseiKakuGo-W5'))
FONT = 'HeiseiKakuGo-W5'
NAVY = colors.HexColor('#203650')
TEAL = colors.HexColor('#087f70')
SLATE = colors.HexColor('#516378')
PALE = colors.HexColor('#f1f5f8')

styles = {
 'title':ParagraphStyle('title',fontName=FONT,fontSize=17,leading=24,textColor=NAVY,spaceAfter=9),
 'subtitle':ParagraphStyle('sub',fontName=FONT,fontSize=10,leading=16,textColor=SLATE,spaceAfter=16),
 'h2':ParagraphStyle('h2',fontName=FONT,fontSize=12,leading=19,textColor=NAVY,spaceBefore=15,spaceAfter=8),
 'body':ParagraphStyle('body',fontName=FONT,fontSize=9.1,leading=16,textColor=NAVY,spaceAfter=8),
 'small':ParagraphStyle('small',fontName=FONT,fontSize=8.2,leading=13.6,textColor=SLATE,spaceAfter=6),
 'th':ParagraphStyle('th',fontName=FONT,fontSize=8.4,leading=13,textColor=colors.white),
 'cell':ParagraphStyle('cell',fontName=FONT,fontSize=8.2,leading=13,textColor=NAVY),
 'footer':ParagraphStyle('footer',fontName=FONT,fontSize=8,leading=12,textColor=SLATE),
}

def P(s,key='body'):
 return Paragraph(escape(s),styles[key])

def table(headers,rows,widths):
 cells=[[P(x,'th') for x in headers]]+[[P(str(x),'cell') for x in row] for row in rows]
 t=Table(cells,colWidths=widths,repeatRows=1,hAlign='LEFT')
 t.setStyle(TableStyle([
     ('BACKGROUND',(0,0),(-1,0),NAVY),
     ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,PALE]),
     ('VALIGN',(0,0),(-1,-1),'MIDDLE'),
     ('LEFTPADDING',(0,0),(-1,-1),10),('RIGHTPADDING',(0,0),(-1,-1),10),
     ('TOPPADDING',(0,0),(-1,-1),9),('BOTTOMPADDING',(0,0),(-1,-1),9),
     ('LINEBELOW',(0,-1),(-1,-1),0.4,colors.HexColor('#cedbe5')),
 ]))
 return t

def footer(canvas,doc):
 canvas.saveState()
 canvas.setStrokeColor(colors.HexColor('#dce3eb'))
 canvas.line(16*mm,16*mm,194*mm,16*mm)
 canvas.setFont(FONT,7.6)
 canvas.setFillColor(SLATE)
 canvas.drawString(17*mm,12*mm,'Physical AI 最終課題  |  Q-learning 2D格子シミュレーション')
 canvas.drawRightString(193*mm,12*mm,str(doc.page))
 canvas.restoreState()

def main():
 data=json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
 rr={r['condition']:r for r in data['scenario_results']}
 f=lambda label:rr[label]
 original=f('Original environment / trained AI')
 off_old=f('Add (3,6) / original AI')
 off_new=f('Add (3,6) / retrained AI')
 on_old=f('Add (6,2) / original AI')
 on_new=f('Add (6,2) / retrained AI')
 random=f('Original environment / random')

 story=[]
 story += [P('Physical AI 最終課題 説明資料','title'),P('Q-learningによる自律移動ロボットの障害物回避と、環境変化への再学習の比較','subtitle')]
 story += [P('1. 課題の目的','h2'),
   P('8×8の2次元格子環境で、ロボットが障害物を避けてスタート (0,0) からゴール (7,7) に移動する課題を設定した。強化学習（Q-learning）で行動方策を獲得し、障害物を増やした場合に、元のAIと再学習したAIの動作がどのように異なるか比較した。'),
   P('本作品は簡易的な離散格子シミュレーションであり、実機やGazeboなどの物理エンジンを使用したものではない。','small')]
 story += [P('2. 開発環境・実装内容','h2'),
   table(['項目','設定'],[
       ('実行環境','Google Colab または PythonのCPU環境'),
       ('開発言語','Python（NumPy、Pillow、FFmpeg）'),
       ('強化学習','表形式のQ-learning、6,000エピソード、乱数シード2026'),
       ('状態と行動','状態＝格子上の位置、行動＝上下左右の4種類'),
       ('報酬','ゴール：+10、障害物／壁：-1、通常移動：-0.08'),
       ('評価','最大40ステップ、固定開始位置での到達可否・ステップ数'),
       ],[44*mm,145*mm])]
 story += [P('3. 実験条件','h2'),
   P('実験A：元の12個の障害物で学習・評価する。実験B：元のAIを保持したまま (3,6) に障害物を1個追加し、再学習したAIとも比較する。実験C：元の経路上にある (6,2) に障害物を追加し、元のAIと再学習したAIを比較する。'),
   P('追加障害物は実験ごとに元の12個の配置へ戻しており、(3,6) と (6,2) を同時には追加していない。','small')]
 story.append(PageBreak())
 story += [P('4. 比較実験の結果','h2')]
 items=[
    ('元の環境／ランダム',random),
    ('元の環境／学習済みAI',original),
    ('(3,6)追加／元のAI',off_old),
    ('(3,6)追加／再学習AI',off_new),
    ('(6,2)追加／元のAI',on_old),
    ('(6,2)追加／再学習AI',on_new),
 ]
 rows=[]
 for label,r in items:
    rows.append((label,f"{r['success_rate_percent']:.0f}%",str(r['one_run_steps']),str(r['one_run_blocked'])))
 story.append(table(['条件','到達率*','行動数','移動不能回数'],rows,[81*mm,30*mm,30*mm,48*mm]))
 story.append(Spacer(1,6*mm))
 story.append(P('*到達率：固定のスタート・ゴールで100回実行した値。学習済みAIは決定論的で、100回の結果は独立した100環境での性能を意味しない。','small'))
 story.append(P('元のAIは14ステップでゴールへ到達した。(3,6) は元の経路上になかったため、追加後も元のAIは14ステップで到達した。一方、(6,2) は元の経路上にあり、追加後は元のAIが同じ方向への移動を繰り返して40ステップ以内に到達できなかった。再学習後のAIは14ステップで到達した。'))
 pic = OUTPUT/'route_comparison.png'
 if pic.exists():
    story.append(Image(str(pic),width=177*mm,height=77.4*mm))
    story.append(P('図：左から、元の環境、(3,6)追加、(6,2)追加。各条件とも追加前に学習した元のAIを評価した軌跡。','small'))
 story += [P('5. 考察','h2'),
   P('今回の結果から、環境変化の位置によって元の行動方策への影響が異なることが分かった。経路外の障害物を増やしても元の行動には影響しなかったが、経路上の障害物が増えると元の行動方策が失敗した。再学習によって別の経路を獲得できたことから、固定環境での再学習の有効性が確認できた。')]
 story.append(PageBreak())
 story += [P('6. 工夫した点','h2'),
  P('① ただ学習済みAIを動かすだけでなく、学習前後および障害物変更前後を同じコードで再現できるようにした。'),
  P('② 経路外 (3,6) と経路上 (6,2) の2条件を分けることで、環境変化の位置によって影響が異なることを評価した。'),
  P('③ 乱数シードを固定し、再学習と元の学習済み方策の比較に同じ環境条件を使用した。'),
  P('④ ColabのCPUで実行可能な軽量な実装とし、評価結果をCSV・動画・画像として保存した。')]
 story += [P('7. 限界と今後の展望','h2'),
  P('本課題は離散的な2次元格子上の実験であり、ロボットアームや車輪の運動学、連続空間、センサー雑音、物理衝突などは表現していない。また、固定したスタート・ゴールでの評価のみなので、未知のマップでも成功するとは言えない。今後は移動開始位置や障害物配置をランダム化し、学習せずに新環境へ適応できるかを評価するとともに、物理シミュレータへの拡張を検討する。')]
 story += [P('8. 提出情報','h2'),
  P('GitHub 公開リポジトリ：' + GITHUB_URL),
  P('YouTube 比較動画：' + YOUTUBE_URL),
  P('利用した学習・実行環境：Google Colab／CPU'),
  P('本コードは先に作成した学習サンプルを基に、生成AIの補助を利用して比較実験の実装・整理を行った。結果は同梱コードの実行出力に基づく。','small')]
 story += [P('提出前の確認','h2'),
  P('・PDFファイル名の「提出前にアカウント名へ変更」を、実際のomnicampusアカウント名に置換する。'),
  P('・動画を再生して内容を確認し、YouTubeの1～3分の要件を満たすことを確認する。'),
  P('・GitHubの公開設定（Public）と提出URLを確認する。')]
 doc=SimpleDocTemplate(str(OUTPDF),pagesize=A4,rightMargin=17*mm,leftMargin=17*mm,
                       topMargin=19*mm,bottomMargin=20*mm,title='Physical AI最終課題 説明資料',author='Physical AI final project')
 doc.build(story,onFirstPage=footer,onLaterPages=footer)

 md=f'''# Physical AI最終課題 説明資料

## Q-learningによる自律移動ロボットの障害物回避と環境変化への対応

### 目的
8×8の2次元格子環境でQ-learningによるロボットの障害物回避を実装する。元の障害物配置に加え、経路外(3,6)、経路上(6,2)のそれぞれに障害物を追加し、変更前のAIと再学習AIを比較した。

### 方法
- Colab（CPU）／Python／NumPy／Pillow／FFmpeg
- 表形式Q-learning、学習6000エピソード、乱数シード2026
- 上下左右4行動。ゴール:+10、壁・障害物:-1、移動:-0.08
- 既存AIを保持したまま地図を変更して評価し、変更後の環境で再学習する。

### 結果
| 条件 | 到達率（固定位置100回反復） | 行動数 | 移動不能回数 |
|---|---:|---:|---:|
'''
 for label,r in items:
    md+=f"| {label} | {r['success_rate_percent']:.0f}% | {r['one_run_steps']} | {r['one_run_blocked']} |\n"
 md+='''
### 考察
(3,6)は元のAIの経路上にないため、元のAIの到達に影響しなかった。一方(6,2)は経路上なので、元のAIが移動不能となったが、再学習したAIは14ステップで到達した。環境変化の位置によって再学習の必要性が変わることを確認した。

### 工夫
- 経路外と経路上に障害物を加えた2条件で比較した。
- 元のAIを再学習で上書きせず保持して、同じ変更後の環境で公平に比較した。
- 乱数シードを固定し、CSV・MP4・静止画像に自動保存した。

### 限界
実機ではなく、離散2D格子の簡易シミュレーション。固定開始位置・固定ゴールの決定論的AIを100回反復した数値であり、未知マップでの一般化性能は示さない。物理エンジン、センサーノイズ、連続座標は未実装。

### 公開情報（提出前に記入）
- GitHub: ［公開URL］
- YouTube: ［今回作成した新しい比較動画URL］
- 作成補助：生成AIを利用
'''
 OUTMD.write_text(md.replace('［公開URL］',GITHUB_URL).replace('［今回作成した新しい比較動画URL］',YOUTUBE_URL),encoding='utf-8')
 print('Generated PDF:',OUTPDF,'size=',OUTPDF.stat().st_size)
 print('Generated editable text:',OUTMD)

if __name__=='__main__':
 main()


## 3. 動画と評価結果の確認
生成された `comparison_demo.mp4` は112秒（1分52秒）です。旧80秒動画とは内容が違います。

In [ ]:
from IPython.display import Video, display, Image as IPyImage
from pathlib import Path
print(Path('results/results.txt').read_text(encoding='utf-8'))
display(IPyImage(filename='results/route_comparison.png', width=900))
display(Video('results/comparison_demo.mp4', embed=True, width=850))

## 4. 提出用ファイルをZIPにまとめて保存
YouTubeには `comparison_demo.mp4`、omnicampusにはPDFとGitHub URL・YouTube URLを提出します。GitHubはPublic設定にします。

In [ ]:
import zipfile
from pathlib import Path
bundle = Path('PhysicalAI_実行結果とレポート.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in Path('results').glob('*'):
        if p.is_file():
            z.write(p, p.as_posix())
    for p in Path('.').glob('説明資料_PAI最終課題_*.pdf'):
        z.write(p, p.name)
    for p in Path('.').glob('レポート文章_*.md'):
        z.write(p, p.name)
print('ZIP作成:', bundle, '容量:', bundle.stat().st_size, 'bytes')
try:
    from google.colab import files
    files.download(str(bundle))
except ImportError:
    print('Colab以外ではZIPを作成した作業フォルダから取得してください。')